In [1]:
!pip install tensorflow numpy

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
#Dataset (sentences + labels)
sentences = [
    ["Ram", "lives", "in", "Pune"],
    ["Shyam", "works", "in", "Mumbai"]
]

labels = [
    ["NNP", "VBZ", "IN", "NNP"],
    ["NNP", "VBZ", "IN", "NNP"]
]

In [4]:
# word tokenizer
word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(sentences)

X = word_tokenizer.texts_to_sequences(sentences)

# label tokenizer
tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(labels)

y = tag_tokenizer.texts_to_sequences(labels)

In [5]:
#Padding
max_len = max(len(s) for s in X)

X = pad_sequences(X, maxlen=max_len, padding='post')
y = pad_sequences(y, maxlen=max_len, padding='post')

# reshape y for model
y = np.expand_dims(y, -1)

In [6]:
#Model
vocab_size = len(word_tokenizer.word_index) + 1
tag_size = len(tag_tokenizer.word_index) + 1

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=8, input_length=max_len))
model.add(LSTM(16, return_sequences=True))
model.add(Dense(tag_size, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
#Training
model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.2500 - loss: 1.3857
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - accuracy: 0.5000 - loss: 1.3844
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.5000 - loss: 1.3832
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.5000 - loss: 1.3819
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.5000 - loss: 1.3806
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.5000 - loss: 1.3793
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - accuracy: 0.5000 - loss: 1.3779
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.5000 - loss: 1.3766
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 623ms/step - accuracy: 0.5000 - loss: 1.3753
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - accuracy: 0.5000 - loss: 1.3739
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - accuracy: 0.5000 - loss: 1.3725
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.5000 - loss: 

In [8]:
#Prediction
test_sentence = ["Ram", "works", "in", "Mumbai"]
test_seq = word_tokenizer.texts_to_sequences([test_sentence])
test_seq = pad_sequences(test_seq, maxlen=max_len, padding='post')

pred = model.predict(test_seq)

predicted_tags = np.argmax(pred, axis=-1)

# reverse mapping
index_to_tag = {v: k for k, v in tag_tokenizer.word_index.items()}

print("Sentence:", test_sentence)
print("Predicted Tags:")
for i, idx in enumerate(predicted_tags[0]):
    if idx != 0:
        print(test_sentence[i], "→", index_to_tag.get(idx))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
Sentence: ['Ram', 'works', 'in', 'Mumbai']
Predicted Tags:
Ram → nnp
works → nnp
in → nnp
Mumbai → nnp
